# SMA Crossover — Trend Following with Parameter Optimization

The SMA crossover is a classic trend-following strategy. The premise: when a fast moving average crosses above a slow moving average, momentum is shifting upward — enter long. When it crosses below — exit.

Its simplicity makes it an excellent vehicle for demonstrating **parameter optimization** and **walk-forward validation** — two techniques that separate rigorous research from curve-fitting.

**Signal logic:**
- fast SMA crosses above slow SMA → BUY
- fast SMA crosses below slow SMA → SELL

---

## Contents
1. Setup & data generation
2. Single-run backtest
3. Grid search parameter optimization
4. Walk-forward validation
5. Overfitting risk — in-sample vs out-of-sample

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import quantcore as qc
from quantcore.analytics import calculate_all_metrics, calculate_returns
from quantcore.plotting import plot_full_tearsheet
from quantcore.walk_forward import GridSearchOptimizer, WalkForwardAnalyzer

print(f'QuantCore {qc.version()}')

## 1. Synthetic Data

We generate a trend-with-noise series: a slow underlying drift plus Gaussian noise plus occasional regime shifts. This gives the crossover strategy something to trade while remaining realistic.

In [ ]:
np.random.seed(7)

N = 1500

# regime-switching trend: alternating bull/bear phases
trend    = np.zeros(N)
phase    = 1            # +1 = bull, -1 = bear
phase_len = 0
for i in range(1, N):
    phase_len += 1
    # switch regime every ~150 bars on average
    if phase_len > 100 and np.random.rand() < 0.01:
        phase     = -phase
        phase_len = 0
    trend[i] = trend[i - 1] + phase * 0.12

noise  = np.cumsum(np.random.randn(N) * 0.8)
prices = 100.0 + trend + noise
prices = np.maximum(prices, 10.0)   # keep prices positive

start_ns   = int(pd.Timestamp('2019-01-02').value)
day_ns     = int(pd.Timedelta('1D').value)
timestamps = [start_ns + i * day_ns for i in range(N)]

bars = [
    qc.BarData(
        'TREND',
        timestamps[i],
        prices[i],
        prices[i] + abs(np.random.randn() * 0.4),
        prices[i] - abs(np.random.randn() * 0.4),
        prices[i],
        1_000_000.0,
    )
    for i in range(N)
]

fig, ax = plt.subplots(figsize=(14, 4))
dates = pd.to_datetime(timestamps, unit='ns')
ax.plot(dates, prices, linewidth=1.2, color='#2E86AB')
ax.set_title('Simulated Regime-Switching Trend Price Series', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Bars: {N}')
print(f'Price range: ${prices.min():.2f} – ${prices.max():.2f}')

## 2. Baseline Backtest — SMA(20/100)

In [ ]:
INITIAL_CAPITAL = 100_000.0

strategy = qc.SMACrossover(fast_period=20, slow_period=100)

results = qc.run_backtest(
    strategy=strategy,
    data={'TREND': bars},
    initial_capital=INITIAL_CAPITAL,
)

equity_curve = np.array(results['equity_curve'])
ts           = np.array(results['timestamps'])
returns      = calculate_returns(equity_curve)
metrics      = calculate_all_metrics(equity_curve)

print(metrics)

In [ ]:
fig = plot_full_tearsheet(
    equity_curve,
    returns,
    timestamps=ts,
    title='SMA Crossover (20/100) — Performance Tearsheet',
)
plt.show()

## 3. Grid Search Parameter Optimization

We use QuantCore's `GridSearchOptimizer` to sweep the full parameter space and rank configurations by Sharpe ratio.

> **Warning:** running grid search on the full dataset and choosing the best parameters is **in-sample optimisation**. Results will be overly optimistic. Section 4 addresses this with walk-forward validation.

In [ ]:
optimizer = GridSearchOptimizer(
    strategy_factory=lambda fast_period, slow_period: qc.SMACrossover(fast_period, slow_period),
    param_grid={
        'fast_period': [5, 10, 15, 20, 30, 50],
        'slow_period': [50, 75, 100, 150, 200],
    },
    metric='sharpe_ratio',
)

opt_results = optimizer.optimize(
    data={'TREND': bars},
    initial_capital=INITIAL_CAPITAL,
    verbose=False,
)

df = optimizer.get_results_dataframe()
print(f'Combinations tested: {len(df)}')
print()
print('Top 10 by Sharpe Ratio:')
print(df.head(10).to_string(index=False))

In [ ]:
# pivot fast vs slow period into a Sharpe heatmap
fast_vals = sorted(df['fast_period'].unique())
slow_vals = sorted(df['slow_period'].unique())

pivot = df.pivot_table(index='fast_period', columns='slow_period', values='sharpe_ratio')

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=-0.5, vmax=1.5)

ax.set_xticks(range(len(slow_vals)))
ax.set_xticklabels([str(v) for v in slow_vals])
ax.set_yticks(range(len(fast_vals)))
ax.set_yticklabels([str(v) for v in fast_vals])
ax.set_xlabel('Slow Period', fontsize=12)
ax.set_ylabel('Fast Period', fontsize=12)
ax.set_title('Grid Search — Sharpe Ratio Heatmap (In-Sample)', fontsize=14, fontweight='bold')

for i, fp in enumerate(fast_vals):
    for j, sp in enumerate(slow_vals):
        val = pivot.loc[fp, sp] if (fp in pivot.index and sp in pivot.columns) else np.nan
        if not np.isnan(val):
            color = 'white' if abs(val) > 0.8 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 4. Walk-Forward Validation

Walk-forward analysis is the standard approach to honest strategy validation:

1. Take an in-sample training window → find best parameters
2. Apply those parameters to the immediately following out-of-sample window
3. Slide both windows forward and repeat

This mimics how a real system would be re-optimised periodically, and prevents the optimizer from 'knowing the future'.

In [ ]:
wf = WalkForwardAnalyzer(
    strategy_factory=lambda fast_period, slow_period: qc.SMACrossover(fast_period, slow_period),
    param_grid={
        'fast_period': [10, 20, 30],
        'slow_period': [75, 100, 150],
    },
    train_size=300,
    test_size=100,
    metric='sharpe_ratio',
)

wf_result = wf.analyze(
    data={'TREND': bars},
    initial_capital=INITIAL_CAPITAL,
)

print(wf_result.summary())

## 5. In-Sample vs Out-of-Sample — The Overfitting Gap

A healthy strategy shows modest degradation moving from in-sample to out-of-sample. A large gap signals overfitting.

In [ ]:
is_sharpes  = [r.sharpe_ratio for r in wf_result.in_sample_results]
oos_sharpes = [r['sharpe_ratio'] for r in wf_result.out_of_sample_results]
windows     = list(range(1, len(is_sharpes) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# per-window IS vs OOS Sharpe
ax = axes[0]
ax.plot(windows, is_sharpes,  marker='o', linewidth=2, label='In-Sample',     color='#2E86AB')
ax.plot(windows, oos_sharpes, marker='s', linewidth=2, label='Out-of-Sample', color='#E84855')
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Window')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('IS vs OOS Sharpe per Window', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# scatter: IS Sharpe predicts OOS Sharpe?
ax = axes[1]
ax.scatter(is_sharpes, oos_sharpes, s=80, color='#3BB273', zorder=5)
for i, (x, y) in enumerate(zip(is_sharpes, oos_sharpes)):
    ax.annotate(f'W{i+1}', (x, y), textcoords='offset points', xytext=(6, 4), fontsize=8)
lim_min = min(min(is_sharpes), min(oos_sharpes)) - 0.2
lim_max = max(max(is_sharpes), max(oos_sharpes)) + 0.2
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=0.8, label='IS = OOS line')
ax.set_xlabel('In-Sample Sharpe')
ax.set_ylabel('Out-of-Sample Sharpe')
ax.set_title('Does IS Sharpe Predict OOS Sharpe?', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

avg_is  = np.mean(is_sharpes)
avg_oos = np.mean(oos_sharpes)
print(f'Average IS Sharpe:  {avg_is:.3f}')
print(f'Average OOS Sharpe: {avg_oos:.3f}')
print(f'Overfitting ratio:  {avg_oos / avg_is:.2f}  (1.0 = no overfitting)')